# 02. 변수 생성 (마스터 데이터셋 생성)

**통합 담당자만 관리하는 노트북.**

각 팀원이 `src/features.py`에 추가한 함수들을 순서대로 호출하여
모든 변수가 포함된 마스터 데이터셋(`apt_master.csv`)을 생성한다.

**작업 순서:**
1. 외부 데이터 7종을 `data/raw/`에 위치시킴 (Google Drive에서 다운로드)
2. 이 노트북 전체 실행
3. 결과 `data/processed/apt_master.csv`를 Google Drive에 공유

**주의:** 카카오 API 호출이 포함된 함수가 있다면 시간이 오래 걸릴 수 있음 (단지 7,000개 × 4 시설 = 약 30분).

## 1. 라이브러리 로드

In [1]:
import pandas as pd
import numpy as np
import sys
import os
import platform
import matplotlib

if platform.system() == 'Darwin':
    matplotlib.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    matplotlib.rcParams['font.family'] = 'Malgun Gothic'
else:
    matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

sys.path.append('../src')
from features import (
    add_transport_features,
    add_infra_features,
    add_negative_features,
)

/Users/beomjun/code/dl/.venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


## 2. 전처리 데이터 로드

In [2]:
df = pd.read_csv('../data/processed/apt_preprocessed.csv', encoding='utf-8-sig')
print(f'전처리 데이터 shape: {df.shape}')
print(f'컬럼: {df.columns.tolist()}')

전처리 데이터 shape: (173399, 15)
컬럼: ['시군구', '단지명', '전용면적(㎡)', '계약년월', '계약일', '거래금액(만원)', '층', '건축년도', '도로명', '계약년도', '구', '강남권', '노후도', '위도', '경도']


## 3. 외부 데이터 로드

각 팀원이 수집한 외부 데이터를 한 곳에서 로드.
컬럼명/인코딩은 데이터에 따라 다르므로 확인 후 통일.

In [3]:
# 교통 (담당: 김채영)
subway = pd.read_csv('../data/raw/서울_지하철.csv', encoding='utf-8-sig')
bus    = pd.read_csv('../data/raw/서울_버스정류장.csv', encoding='cp949')

# 인프라 (담당: 임종욱)
park          = pd.read_csv('../data/raw/서울_공원.csv', encoding='cp949')
academy       = pd.read_csv('../data/raw/서울_학원.csv', encoding='cp949')
hospital      = pd.read_csv('../data/raw/서울_의료시설.csv', encoding='cp1006')

# 부정 환경 (담당: 이재령)
entertainment = pd.read_csv('../data/raw/서울_유흥주점.csv', encoding='cp949')
motel         = pd.read_csv('../data/raw/서울_모텔.csv', encoding='cp949')

# 각 데이터 shape 확인
for name, d in [('subway', subway), ('bus', bus),
                ('park', park), ('academy', academy), ('hospital', hospital),
                ('entertainment', entertainment), ('motel', motel)]:
    print(f'{name:>15}: {d.shape}')

         subway: (407, 5)
            bus: (16980, 9)
           park: (1761, 19)
        academy: (45079, 39)
       hospital: (18998, 39)
  entertainment: (1652, 5)
          motel: (1856, 5)


## 4. 외부 데이터 정제

각 외부 데이터의 위도/경도 컬럼명을 통일 (`위도`, `경도`로).
실제 컬럼명은 데이터에 따라 다르므로 필요시 rename.

In [4]:
# subway 컬럼명 통일
subway = subway.rename(columns={'역위도': '위도', '역경도': '경도'})

# academy 헤더 없이 다시 로드 (마지막 두 컬럼이 경도/위도)
academy = pd.read_csv('../data/raw/서울_학원.csv', encoding='cp949', header=None)
academy.columns = [f'col_{i}' for i in range(len(academy.columns))]
academy = academy.rename(columns={
    f'col_{len(academy.columns)-2}': '경도',
    f'col_{len(academy.columns)-1}': '위도'
})

# hospital 헤더 없이 다시 로드
hospital = pd.read_csv('../data/raw/서울_의료시설.csv', encoding='cp1006', header=None)
hospital.columns = [f'col_{i}' for i in range(len(hospital.columns))]
hospital = hospital.rename(columns={
    f'col_{len(hospital.columns)-2}': '경도',
    f'col_{len(hospital.columns)-1}': '위도'
})

In [5]:
# 결측 좌표 제거
for d_name, d in [('subway', subway), ('bus', bus),
                  ('park', park), ('academy', academy), ('hospital', hospital),
                  ('entertainment', entertainment), ('motel', motel)]:
    before = len(d)
    d.dropna(subset=['위도', '경도'], inplace=True)
    print(f'{d_name}: {before} → {len(d)}건')

subway: 407 → 407건
bus: 16980 → 16980건
park: 1761 → 1761건
academy: 45080 → 45080건
hospital: 18999 → 18999건
entertainment: 1652 → 1652건
motel: 1856 → 1856건


## 5. 변수 생성 (각 팀원 함수 호출)

In [6]:
# 교통 변수 추가
df = add_transport_features(df, subway_df=subway, bus_df=bus)

# 교육 변수 추가
df = add_infra_features(df, park_df=park, academy_df=academy, hospital_df=hospital)

# 부정 변수 추가
df = add_negative_features(df, entertainment_df=entertainment, motel_df=motel)

print(f'\n변수 추가 완료. 최종 shape: {df.shape}')
print(f'컬럼: {df.columns.tolist()}')

[subway] 변수 추가 완료: subway_nearest_dist, subway_count_500m
[bus] 변수 추가 완료: bus_nearest_dist, bus_count_500m
[park] 변수 추가 완료: count_park_nearby
[academy] 변수 추가 완료: count_academy_nearby
[hospital] 변수 추가 완료: count_hospital_nearby
[vice] 변수 추가 완료: vice_nearest_dist, vice_count_500m

변수 추가 완료. 최종 shape: (173399, 23)
컬럼: ['시군구', '단지명', '전용면적(㎡)', '계약년월', '계약일', '거래금액(만원)', '층', '건축년도', '도로명', '계약년도', '구', '강남권', '노후도', '위도', '경도', 'subway_nearest_dist', 'subway_count_500m', 'bus_nearest_dist', 'bus_count_500m', 'count_park_nearby', 'count_academy_nearby', 'count_hospital_nearby', 'vice_count_500m']


## 6. 결측치 확인 및 처리

In [7]:
# 변수별 결측치 확인
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print('결측치 발생 컬럼:')
    print(missing)
else:
    print('결측치 없음')

결측치 없음


In [8]:
# 좌표 변환 실패 등으로 변수 결측인 행 제거 (선택)
# 결측 비율이 낮으면 제거, 높으면 0/평균 대체 고려
before = len(df)
df = df.dropna()
print(f'결측 행 제거: {before} → {len(df)}건')

결측 행 제거: 173399 → 173399건


## 7. 마스터 데이터셋 저장

In [9]:
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/apt_master.csv', index=False, encoding='utf-8-sig')

print('저장 완료: ../data/processed/apt_master.csv')
print(f'최종 shape: {df.shape}')
print(f'\n이 파일을 Google Drive에 업로드하여 팀원들과 공유.')

저장 완료: ../data/processed/apt_master.csv
최종 shape: (173399, 23)

이 파일을 Google Drive에 업로드하여 팀원들과 공유.


## 8. 마스터 데이터셋 요약

다음 노트북(03_experiments.ipynb)에서 사용할 변수 목록 정리.

In [10]:
# 변수 그룹별 컬럼 정리 (03 노트북 슬롯 구성용)
PHYSICAL    = ['전용면적(㎡)', '층', '노후도']
REGION      = ['강남권']
TRANSPORT   = [c for c in df.columns if c.startswith(('subway_', 'bus_'))]
INFRA = [c for c in df.columns if c.startswith('count_') and c.endswith('_nearby')]
NEGATIVE    = [c for c in df.columns if c.startswith('vice_')]

print('=== 변수 그룹 요약 ===')
print(f'PHYSICAL  ({len(PHYSICAL)}): {PHYSICAL}')
print(f'REGION    ({len(REGION)}): {REGION}')
print(f'TRANSPORT ({len(TRANSPORT)}): {TRANSPORT}')
print(f'INFRA     ({len(INFRA)}): {INFRA}')
print(f'NEGATIVE  ({len(NEGATIVE)}): {NEGATIVE}')

=== 변수 그룹 요약 ===
PHYSICAL  (3): ['전용면적(㎡)', '층', '노후도']
REGION    (1): ['강남권']
TRANSPORT (4): ['subway_nearest_dist', 'subway_count_500m', 'bus_nearest_dist', 'bus_count_500m']
INFRA     (3): ['count_park_nearby', 'count_academy_nearby', 'count_hospital_nearby']
NEGATIVE  (1): ['vice_count_500m']
